In [ ]:
!pip install torch transformers datasets accelerate bitsandbytes

In [ ]:
!pip install -q -U trl==0.8.6 peft==0.11.1

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
model_name = "Qwen/Qwen2-1.5B-Instruct"

#Q-loRA

# 1- Quantization(Q)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Enable 4-bit loading
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

#Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # Fix padding issue for training

#Load the Base Model with Quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, # Apply the 4-bit config
    device_map="auto", # Automatically use the GPU
    trust_remote_code=True
)
model.config.use_cache = False

#2 - LORA
peft_config = LoraConfig(
    r=16,#Size of the tiny matrices
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear" # Attach adapters to ALL linear layers
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
#Load and Shuffle Dataset
dataset = load_dataset("flytech/python-codes-25k", split="train")
if "text" in dataset.column_names:
    print("deleting text col")
    dataset = dataset.remove_columns("text")

dataset = dataset.shuffle(seed=42).select(range(2500))
print("sample of dataset:")
print(dataset[0])

#Formatting Function
# We transform the raw "Input/Output" into a Qwen Chat format
def formatting_prompts_func(examples):
    output_texts = []

    #instruction => question , input => context , output => code solution
    for instruction, input_text, output in zip(examples['instruction'], examples['input'], examples['output']):

        system_prompt = "You are a helpful coding assistant expert in Python."

        user_content = instruction
        if input_text: #if there is data, append it to instruction
            user_content += f"\nInput: {input_text}"

        output = output.replace("```python", "").replace("```", "").strip()

        # format: <|im_start|>role\nContent<|im_end|>
        text = (
            f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\n{user_content}<|im_end|>\n"
            f"<|im_start|>assistant\n{output}<|im_end|>"
        )

        output_texts.append(text)

    return output_texts

training_args = TrainingArguments(
    output_dir="./part2_results",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate = 2 * 10**-4,
    optim="paged_adamw_8bit",
    #-----------------------
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",          # Standard for LLMs
    report_to="tensorboard",             # Logs metrics for you to see
    gradient_checkpointing=True, # Slower Speed for More Memory.
    group_by_length=True,#sorting
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    max_seq_length=1024,
    args=training_args,
    formatting_func=formatting_prompts_func,
)


deleting text col
sample of dataset:
{'output': '```python\nimport random\n\n# generating a list of unique numbers from 0 to 9 in random order\nrandom_numbers = random.sample(range(0, 10), 10)\n\n# sort list of numbers \nrandom_numbers.sort()\n\n# print sorted list of random numbers\nprint(random_numbers)\n# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]\n```', 'instruction': 'Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order', 'input': ''}


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


In [ ]:
print("Starting training...")
trainer.train()

final_model = "final model"

print(f"Saving model to {final_model}...")
trainer.model.save_pretrained(final_model)
tokenizer.save_pretrained(final_model)

print("DONE! Model saved successfully.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...)


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.280000
20,1.021400
30,0.824700
40,0.861500
50,0.888200
60,0.879400
70,0.779400
80,0.757200
90,0.790100
100,0.909900


Saving model to final model...
DONE! Model saved successfully.


In [19]:
import torch

# 1. Switch model to evaluation mode
model.eval()

# 2. Define the prompt
prompt = "Create a python program to calculate Fibonacci sequence."
formatted_prompt = f"<|im_start|>system\nYou are a helpful coding assistant expert in Python.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

# 3. Tokenize
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

# 4. Generate Code
print("Generating Python code...")
with torch.autocast("cuda"):
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.6,
        pad_token_id=tokenizer.eos_token_id
    )

# 5. Print the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n=== MODEL'S CODE ===")
print(decoded_output.split("assistant")[-1].strip())

Generating Python code...

=== MODEL'S CODE ===
def fibonacci(n):
    if n <= 0:
        return "Incorrect input"
    elif n == 1:
        return 0
    elif n == 2:
        return 1
    else:
        return fibonacci(n-1) + fibonacci(n-2)

print(fibonacci(6)) # Output: 8
